In [1]:
import sys

# jupyter kernels use the notebook's own directory as the cwd, not wherever
# jupyter lab was launched from, so the repo root needs to be added manually
# for `from src...` imports to resolve.
sys.path.append("../")

# Inspect an existing model

In [6]:
from src.modeling.utils import load_mlflow_model

pipeline, version = load_mlflow_model(target="fantasy_points_ppr", model_type="random_forest", tracking_dir="../mlruns")
#print(f"pipeline: {pipeline}")
print(f"Loaded version {version}")

model = pipeline['model']
print(f"model: {model}")

Loaded version 6
model: RandomForestRegressor(oob_score=True)


/Users/vivek.sivakumar/Code/personal/ff-model/notebooks/../src/modeling/utils.py:81: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  model_version = client.get_latest_versions(registered_name, stages=["None"])[0].version


# View the training data

In [7]:
from src.modeling.tabular_models import TabularModel

tab_model = TabularModel(data_dir="../data", tracking_dir="../mlruns", target="fantasy_points_ppr", excluded_features=["fantasy_points*"])
datasets = tab_model.split_data()

train_df = datasets["X_train"]
train_df.head(10)

2026-08-05 14:55:45,522 - src.modeling.tabular_models - INFO - load_data - Loaded data: 11152 rows


,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_first_downs,passing_2pt_conversions,passing_air_yards,passing_yards_after_catch,sacks_suffered,...,receiving_10_shrunk_avg,receiving_16_shrunk_avg,receiving_20_shrunk_avg,receiving_40_shrunk_avg,receiving_epa_shrunk_avg,racr_shrunk_avg,target_share_shrunk_avg,air_yards_share_shrunk_avg,wopr_shrunk_avg,seasons_since_played
0,343.0,569.0,4436.0,36.0,15.0,208.0,0.0,394.0,4290.0,50.0,...,0.054878,0.018293,0.018293,0.000000,NaN,NaN,0.000387,0.000000,0.000581,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.913551,2.102804,1.324766,0.077103,2.739741,NaN,0.034618,0.020426,0.066226,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.663551,2.602804,1.574766,0.077103,4.308610,12.112412,0.042770,0.024394,0.081231,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,21.229167,11.508333,7.866667,1.770833,14.297021,11.352537,0.117561,0.155746,0.285364,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,7.666071,3.823214,2.532143,0.346429,4.154489,-12.547834,0.062775,-0.005641,0.090214,0
5,305.0,537.0,3985.0,19.0,21.0,184.0,0.0,319.0,3982.0,55.0,...,0.054878,0.018293,0.018293,0.000000,NaN,NaN,0.000387,0.000000,0.000581,0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,13.479167,7.008333,4.866667,1.020833,8.433756,9.267339,0.087576,0.128700,0.221454,0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.979167,9.508333,7.116667,1.270833,14.093372,13.450451,0.148522,0.154113,0.330662,0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.666071,1.573214,1.032143,0.096429,1.388346,2.313277,0.030447,0.001282,0.046568,0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,19.479167,9.508333,6.866667,2.020833,13.004520,11.246346,0.135728,0.149413,0.308180,0


# Feature Importance

In [8]:
import pandas as pd

def get_feat_importance(model, train_df):
    return pd.DataFrame({'feature':train_df.columns, 'importance':model.feature_importances_}).sort_values('importance', ascending=False)

def plot_feature_importance(fi, num_f=30):
    return fi[:num_f].plot('feature', 'importance', 'barh', figsize=(12,7), legend=False)

In [ ]:
feature_imp = get_feat_importance(model=model, train_df=train_df)

plot_feature_importance(fi=feature_imp, num_f=40)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

PartialDependenceDisplay.from_estimator(model, train_df, train_df.columns[-5:-2])


## What other columns could be features?

In [ ]:
# play by play data
import nflreadpy as nfl

pbp = nfl.load_pbp([2025]).to_pandas()
print(f"pbp columns: {pbp.columns.tolist()}")

In [ ]:
player = nfl.load_player_stats([2025]).to_pandas()
print(f"player stats columns: {player.columns.tolist()}")

In [ ]:
team = nfl.load_team_stats([2025]).to_pandas()
print(f"team stats columns: {team.columns.tolist()}")

In [ ]:
snaps = nfl.load_snap_counts([2025]).to_pandas()
print(f"snap counts columns: {snaps.columns.tolist()}")

In [ ]:
injuries = nfl.load_injuries([2025]).to_pandas()
print(f"injuries columns: {injuries.columns.tolist()}")

In [ ]:
draft = nfl.load_draft_picks([2020]).to_pandas()
print(f"draft picks columns: {draft.columns.tolist()}")

In [ ]:
ff_opportunity = nfl.load_ff_opportunity([2025]).to_pandas()
print(f"team ff_opportunity columns: {ff_opportunity.columns.tolist()}")

## Rankings!

In [ ]:
from src.rankings import Rankings

ranker = Rankings(year=2025, target_col="ppr_fantasy_points_per_game")
rankings = ranker.build_rankings()

ranker.plot_rankings(rankings)

# 

# 

# 